In [1]:
import numpy as np

def load_data_halo(
    sim_path,
    string,
    mass_limit=1,
    mass_width=0.5,
    remove_subhalos=True,
):
    """
    Load halo positions, velocities, and masses.

    Parameters
    ----------
    sim_path : str
        Path to the simulation / halo catalogue.
    string : {"+", "bin"}
        Selection mode passed to halo_selection.
    mass_limit : float, optional
        Mass threshold or bin center, depending on selection mode.
    mass_width : float, optional
        Width of the mass bin when string == "bin".
    remove_subhalos : bool, optional
        If True, keep only parent/host halos with PID == -1.
        Assumes PID is the last column of halo_all.

    Returns
    -------
    pos : ndarray, shape (N, 3), dtype float32
        Halo positions.
    vel : ndarray, shape (N, 3), dtype float32
        Halo velocities.
    mass : ndarray, shape (N,), dtype float32
        Halo masses.
    """

    if string == "+":
        halo_all = halo_selection(sim_path, "+", mass_limit)

    elif string == "bin":
        halo_all = halo_selection(sim_path, "bin", mass_limit, mass_width)

    else:
        raise ValueError(
            f"Unknown string='{string}'. Expected '+' or 'bin'."
        )

    halo_all = np.asarray(halo_all)

    if halo_all.ndim != 2:
        raise ValueError(
            f"halo_selection returned an array with shape {halo_all.shape}; expected 2D."
        )

    if halo_all.shape[1] < 8 and remove_subhalos:
        raise ValueError(
            "remove_subhalos=True requires a PID column, assumed to be the last column."
        )

    if remove_subhalos:
        pid = halo_all[:, -1].astype(np.int64)
        host_mask = pid == -1
        halo_all = halo_all[host_mask]

    pos = halo_all[:, :3]
    vel = halo_all[:, 3:6]
    mass = halo_all[:, 6]

    return (
        pos.astype(np.float32),
        vel.astype(np.float32),
        mass.astype(np.float32),
    )

In [ ]:
import numpy as np

def halo_selection(
    data_address,
    string,
    mass_limit,
    mass_width=0.5,
    remove_subhalos=True,
    extra_columns=False,
    return_counts=False,
):
    """ 
    Filter a Rockstar halo catalogue.

    Parameters
    ----------
    data_address : str
        Path to the halo catalogue.
    string : {"+", "-", "bin", "all"}
        Selection mode:
        "+"   : select halos with M200b > mass_limit
        "-"   : select halos with M200b <= mass_limit
        "bin" : select halos in a mass bin around mass_limit
        "all" : select all halos
    mass_limit : float
        Mass threshold or bin center.
    mass_width : float, optional
        Width parameter for the mass bin.
    remove_subhalos : bool, optional
        If True, remove subhalos by requiring PID == -1.
        Assumes PID is the last column of a parent-processed catalogue.
    extra_columns : bool, optional
        If True, append Rvir as an extra column.
    return_counts : bool, optional
        If True, return halo_pop, num_halos, tot_halos.

    Returns
    -------
    halo_pop : ndarray
        Columns:
        [X, Y, Z, VX, VY, VZ, M200b]
        or
        [X, Y, Z, VX, VY, VZ, M200b, Rvir]
        if extra_columns=True.
    """

    data = np.loadtxt(data_address)

    # Ensure 2D even if the catalogue has only one halo
    data = np.atleast_2d(data)

    tot_halos_before_subhalo_cut = data.shape[0]

    # Remove subhalos if requested.
    # PID is assumed to be the last column of out_*_sub.list.
    if remove_subhalos:
        pid = data[:, -1].astype(np.int64)
        data = data[pid == -1]

    tot_halos_after_subhalo_cut = data.shape[0]

    # M200b column in Rockstar catalogue
    M200b = data[:, 20]

    if string == "+":
        condition = M200b > mass_limit

    elif string == "-":
        condition = M200b <= mass_limit

    elif string == "bin":
        exponent = np.floor(np.log10(mass_limit))
        coefficient = mass_limit / (10**exponent)

        if np.isclose(coefficient, 1.0):
            lower = (coefficient - 0.1 * mass_width) * 10**exponent
            upper = (coefficient + mass_width) * 10**exponent
        else:
            lower = (coefficient - mass_width) * 10**exponent
            upper = (coefficient + mass_width) * 10**exponent

        condition = (M200b >= lower) & (M200b <= upper)

    elif string == "all":
        condition = np.ones(data.shape[0], dtype=bool)

    else:
        raise ValueError(
            f"Unknown string={string!r}. Expected '+', '-', 'bin', or 'all'."
        )

    selected = data[condition]

    ncols_out = 8 if extra_columns else 7
    halo_pop = np.zeros((selected.shape[0], ncols_out), dtype=np.float64)

    # X,Y,Z,VX,VY,VZ are columns 8 through 13
    halo_pop[:, 0:6] = selected[:, 8:14]

    # M200b is column 20
    halo_pop[:, 6] = selected[:, 20]

    # Optional Rvir is column 5, in kpc/h
    if extra_columns:
        halo_pop[:, 7] = selected[:, 5]

    if return_counts:
        num_halos = halo_pop.shape[0]
        return halo_pop, num_halos, tot_halos_before_subhalo_cut, tot_halos_after_subhalo_cut

    return halo_pop

In [2]:
sim_path = "./../../simulations/L_1024_Ngrid_2600/0.0ev/output/halos/out_2_sub.list"

halo_all = halo_selection(sim_path, "+", 1)

pid = halo_all[:, -1].astype(np.int64)

print("Total halos:", len(pid))
print("Hosts PID=-1:", np.sum(pid == -1))
print("Subhalos PID!=-1:", np.sum(pid != -1))
print("Unique PID examples:", np.unique(pid[:100]))

NameError: name 'halo_selection' is not defined